In [140]:
import cvxpy as cp
import numpy as np
from scipy.optimize import minimize

# 定义初始值
FLOPS = 312  # 混合精度（Tensor Core）FP16
s_r = 100
h = 5120
L = 40  # layers
V = 32000
epsilon = 10
SLO = 0.0001
M_W = 26
M_G = 40
R = 100

# 生成随机 s_r 数组, 长度为R，每个值的大小为100~1000
s_r = np.random.randint(100, 1000, R)
print("s_r:", s_r)

s_r: [244 792 265 165 111 260 452 282 104 759 848 423 226 729 787 739 695 212
 773 862 310 325 187 751 560 520 186 386 760 736 237 935 248 552 202 706
 826 363 282 920 569 869 613 324 582 330 510 385 803 160 682 589 612 727
 298 997 681 401 315 404 613 386 789 596 113 935 299 157 971 941 292 415
 871 895 663 949 754 316 923 293 792 360 674 801 248 787 568 260 178 972
 398 512 896 209 184 286 760 765 793 695]


In [147]:
I=4
# 定义T_r(l_i)
def T_r(l_i, r):
    A_r = 24 * s_r[r] * h**2 + 4 * s_r[r]**2 * h
    B_r = 24 * h**2 + 4 * h - 20 * s_r[r] * h**2 - 4 * s_r[r]**2 * h
    C_r = 2 * s_r[r] * h * V + 2 * h * V + 2 * epsilon
    return A_r * L + B_r * l_i + C_r

# 定义目标函数
def objective(vars):
    b_i = vars[:I].astype(int)  # 前I个变量是b_i
    l_i = vars[I:]  # 后I个变量是l_i
    obj = 0
    for i in range(I):
        sum_T_r = sum(T_r(l_i[i], r) for r in range(b_i[i]))
        obj += b_i[i] * FLOPS / sum_T_r
    return -obj  # 最大化问题转化为最小化问题

# 定义约束条件
def constraint1(vars, i):
    b_i = vars[:I].astype(int)
    l_i = vars[I:]
    sum_T_r = sum(T_r(l_i[i], r) for r in range(b_i[i]))
    return SLO - sum(1 / FLOPS * sum(T_r(l_i[k], r) for r in range(b_i[k])) for k in range(i + 1))

def constraint2(vars, i):
    b_i = vars[:I].astype(int)
    l_i = vars[I:]
    return min((M_G - M_W) / (2 * h * sum(s_r[r] for r in range(b_i[i]))), L) - l_i[i]

def constraint3(vars):
    b_i = vars[:I].astype(int)
    return np.sum(b_i) - R

# 初始猜测值（加入随机性）
initial_guess = np.concatenate((np.full(I, R // I), np.random.uniform(2, L, I)))  # b_i均匀分配，l_i随机初始化

# 约束条件
constraints = []
for i in range(I):
    constraints.append({'type': 'ineq', 'fun': lambda vars, i=i: constraint1(vars, i)})
    constraints.append({'type': 'ineq', 'fun': lambda vars, i=i: constraint2(vars, i)})
constraints.append({'type': 'eq', 'fun': constraint3})

# 边界条件
bounds = [(1, None)] * I + [(1, L)] * I  # b_i >= 1, l_i >= 1

# 优化
result = minimize(objective, initial_guess, method='SLSQP', bounds=bounds, constraints=constraints)

# 输出结果
b_i_optimal = result.x[:I].astype(int)
l_i_optimal = result.x[I:]
print("Optimal b_i:", b_i_optimal)
print("Optimal l_i:", l_i_optimal)
print("Objective value:", '{:.10f}'.format(-result.fun))
print("Constraints:", [c['fun'](result.x) for c in constraints])

Optimal b_i: [25 25 25 25]
Optimal l_i: [12.40148211 21.76673654 21.39884545 13.07106335]
Objective value: 0.0000000002
Constraints: [-735975108860.4319, -12.401481994419624, -1281233388424.0278, -21.76673641994091, -1833983511636.185, -21.398845331652108, -2556323068771.688, -13.071063234875126, 0]


In [148]:
# 定义T_r(l_i)
def T_r(l_i, r):
    A_r = 24 * s_r[r] * h**2 + 4 * s_r[r]**2 * h
    B_r = 24 * h**2 + 4 * h - 20 * s_r[r] * h**2 - 4 * s_r[r]**2 * h
    C_r = 2 * s_r[r] * h * V + 2 * h * V + 2 * epsilon
    return A_r * L + B_r * l_i + C_r

# 定义优化变量
l_i = cp.Variable(I, nonneg=True)  # l_i是非负实数变量

# 定义b_i为常数
b = R // I  # 因为所有b_i相同，所以b_i = R / I

# 定义目标函数
# 使用cvxpy.inv_pos来处理分数形式
# objective = cp.Maximize(cp.sum([b * FLOPS * cp.inv_pos(cp.sum([T_r(l_i[i], r) for r in range(b)])) for i in range(I)]))
# 将目标函数改成对数形式
objective = cp.Maximize(cp.sum([cp.log(b * FLOPS) - cp.log(cp.sum([T_r(l_i[i], r) for r in range(b)])) for i in range(I)]))

# 定义约束条件
constraints = []

# 约束1
for i in range(I):
    sum_T_r = cp.sum([T_r(l_i[i], r) for r in range(b)])
    constraints.append(1 / FLOPS * sum_T_r <= SLO - cp.sum([1 / FLOPS * cp.sum([T_r(l_i[k], r) for r in range(b)]) for k in range(i)]))

# 约束2
for i in range(I):
    constraints.append(l_i[i] <= cp.minimum((M_G - M_W) / (2 * h * cp.sum([s_r[r] for r in range(b)])), L))

# 优化问题
problem = cp.Problem(objective, constraints)

# 求解
problem.solve(solver=cp.SCS)

# 输出结果
print("Optimal l_i:", l_i.value)
print("Maximum Objective Value:", problem.value)
print("b_i (constant):", b)

DCPError: Problem does not follow DCP rules. Specifically:
The objective is not DCP, even though each sub-expression is.
You are trying to maximize a function that is convex.